# Phase 4 — 입출력 및 캘리브레이션 (예제)

이미지를 읽고 저장하는 법, 좌표계, 그리고 픽셀 → nm 캘리브레이션을 익힙니다. 각 셀을 순서대로 실행하십시오.

## 0. 그래프 한글 폰트 설정

그래프 제목·축 라벨에 한글을 쓸 때 깨지지 않도록, 설치된 한글 폰트를 찾아 적용합니다. (없으면 라벨은 영어로 표기)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
for _n in ['Malgun Gothic','NanumGothic','AppleGothic','Noto Sans CJK KR','Noto Sans KR']:
    if _n in {f.name for f in fm.fontManager.ttflist}:
        plt.rcParams['font.family'] = _n; print('한글 폰트 적용:', _n); break
else:
    print('한글 폰트를 찾지 못했습니다 — 라벨은 영어로 표기합니다.')
plt.rcParams['axes.unicode_minus'] = False

## 1. 이미지 읽기와 저장 (Pillow · OpenCV)
Pillow 와 OpenCV 모두 이미지를 NumPy 배열로 읽어옵니다. 여기서는 예제 이미지를 만들어 저장하고 다시 읽어 확인합니다.

In [ ]:
import numpy as np, cv2
from PIL import Image
import os, tempfile

# 예제 이미지 생성 (가로 띠 + 내부 어두운 층)
img = np.full((120, 200), 180, dtype=np.uint8)
img[50:70, :] = 90

tmp = tempfile.gettempdir()
path = os.path.join(tmp, 'sample.png')

cv2.imwrite(path, img)                 # OpenCV 로 저장
re_cv = cv2.imread(path, cv2.IMREAD_GRAYSCALE)   # OpenCV 로 읽기
re_pil = np.array(Image.open(path))    # Pillow 로 읽기

print('저장 경로 :', path)
print('OpenCV 로 읽음 shape :', re_cv.shape)
print('Pillow 로 읽음 shape :', re_pil.shape)

## 2. 좌표계: img[행, 열] = [y, x]
NumPy 는 **[행, 열]** 순서입니다. 행은 세로(y), 열은 가로(x)이며, 흔히 쓰는 (x, y) 와 순서가 반대입니다.

In [ ]:
g = np.arange(20).reshape(4, 5)   # 4행 5열
print(g)

print('\nimg[1, 3] =', g[1, 3], '  (행 1, 열 3 — 올바름)')
print('img[3, 1] =', g[3, 1], '  (행 3, 열 1 — 좌표를 바꾸면 다른 픽셀)')

## 3. bit depth: 값의 범위
픽셀 값의 범위는 자료형에 따라 다릅니다. 8bit(uint8)은 0~255, 16bit(uint16)은 0~65535 이며 TEM 은 16bit 가 흔합니다.

In [ ]:
a8  = np.array([0, 128, 255], dtype=np.uint8)
a16 = np.array([0, 30000, 65535], dtype=np.uint16)

print('uint8  최대 :', np.iinfo(np.uint8).max)    # 255
print('uint16 최대 :', np.iinfo(np.uint16).max)   # 65535
print('dtype 확인 :', a8.dtype, a16.dtype)

## 4. 표시와 히스토그램
이미지를 표시하고, 히스토그램으로 밝기 분포를 봅니다. 분포는 이후 이진화 임계값을 정할 때 활용합니다.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].imshow(img, cmap='gray', vmin=0, vmax=255)
ax[0].set_title('image'); ax[0].axis('off')
ax[1].hist(img.ravel(), bins=256, range=(0, 255))
ax[1].set_title('histogram')
plt.tight_layout(); plt.show()

## 5. 캘리브레이션: 픽셀 → nm
측정 결과는 픽셀 단위입니다. `scale`(nm/pixel)을 곱하면 실제 길이(nm)가 됩니다.

> 실제 길이(nm) = 픽셀 수(px) × scale(nm/pixel)

In [ ]:
def px_to_nm(px, scale):
    return px * scale

scale = 0.25          # nm/pixel (다음 셀에서 파일로부터 자동 추출)
thickness_px = 80
print('두께 :', px_to_nm(thickness_px, scale), 'nm')   # 20.0

# 면적은 scale 을 두 번 곱합니다 (가로 x 세로)
area_px = 1000
print('면적 :', area_px * scale**2, 'nm^2')           # 62.5

## 6. hyperspy 로 TIFF 의 scale 자동 추출
TEM 으로 저장한 TIFF 에는 픽셀 scale 이 메타데이터로 기록되어 있습니다. hyperspy 가 이 값을 읽어줍니다.
수동 입력 대신 파일에서 직접 읽으므로 휴먼에러가 줄어듭니다.

아래는 실제 TIFF 가 있을 때의 코드입니다. `tif_path` 를 실제 파일 경로로 바꿔 실행하십시오. (hyperspy 미설치 또는 파일이 없으면 안내만 표시되고, 기본 scale 값으로 계속 진행합니다.)

In [ ]:
tif_path = None      # 예: r'D:/data/sample.tif'  로 변경

scale, units = 0.25, 'nm'    # 기본값 (파일을 못 읽을 때 사용)

try:
    import hyperspy.api as hs
    if tif_path:
        sig = hs.load(tif_path)
        ax = sig.axes_manager[-1]
        scale, units = ax.scale, ax.units
        print('파일에서 읽은 scale :', scale, units)
    else:
        print('tif_path 가 None 입니다 — 기본 scale 사용:', scale, units)
except ModuleNotFoundError:
    print('hyperspy 미설치 — 기본 scale 사용:', scale, units)
except Exception as e:
    print('파일을 읽지 못했습니다(', e, ') — 기본 scale 사용:', scale, units)

# 이후 측정에서는 이 scale 을 그대로 사용
print('80 px =', px_to_nm(80, scale), units)

---
예제 실행을 마친 후 `04_practice.ipynb` 의 문제를 해결하십시오.